# FRC Elastic LED Layout Generator

自动生成 Elastic LED 单色视图布局，支持自定义参数。

In [4]:
import json
import ipywidgets as widgets
from IPython.display import display, JSON


In [ ]:
# ========== 从 Constants.java 读取的默认值 ==========
DEFAULT_LED_COUNT = 35
DEFAULT_ZONES = [
    {"name": "ChassisLeft",  "start": 0,  "end": 5},
    {"name": "ShooterMid",   "start": 6,  "end": 28},
    {"name": "ChassisRight", "start": 29, "end": 34},
]

In [ ]:
# ========== 参数面板 ==========
w_led_count = widgets.IntSlider(
    value=DEFAULT_LED_COUNT, min=1, max=100, step=1,
    description='LED 数量:', style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

w_width = widgets.FloatSlider(
    value=50, min=10, max=100, step=5,
    description='单个宽度:', style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

w_height = widgets.FloatSlider(
    value=130, min=20, max=200, step=5,
    description='单个高度:', style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

w_gap = widgets.FloatSlider(
    value=-15, min=-15, max=10, step=1,
    description='间距:', style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

w_zone_gap = widgets.FloatSlider(
    value=0, min=-15, max=50, step=5,
    description='区域间隔:', style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

w_y_offset = widgets.FloatSlider(
    value=-65, min=-200, max=200, step=5,
    description='Y 偏移:', style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

w_period = widgets.FloatSlider(
    value=0.06, min=0.01, max=0.5, step=0.01,
    description='刷新周期:', style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

w_topic_prefix = widgets.Text(
    value='/SmartDashboard/LED/',
    description='Topic 前缀:', style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

output = widgets.Output()

ui = widgets.VBox([
    widgets.HTML('<h3>LED Layout 参数</h3>'),
    w_led_count, w_width, w_height, w_gap, w_zone_gap, w_y_offset,
    w_period, w_topic_prefix,
    widgets.Button(description='生成布局', button_style='primary',
                  layout=widgets.Layout(width='150px')),
    output
])
display(ui)

In [ ]:
# ========== 自动生成 ==========
def generate_layout(btn=None):
    output.clear_output()
    led_count = w_led_count.value
    w = w_width.value
    h = w_height.value
    gap = w_gap.value
    zone_gap = w_zone_gap.value
    y = w_y_offset.value
    period = w_period.value
    prefix = w_topic_prefix.value

    # 自动划分区域：尽量均匀分，每区域最多 8 个 LED
    zones = []
    i = 0
    zone_idx = 0
    zone_names = ['ZoneA', 'ZoneB', 'ZoneC', 'ZoneD', 'ZoneE',
                  'ZoneF', 'ZoneG', 'ZoneH', 'ZoneI', 'ZoneJ']
    while i < led_count:
        chunk_size = min(8, led_count - i)
        name = zone_names[zone_idx] if zone_idx < len(zone_names) else f'Zone{zone_idx}'
        zones.append({"name": name, "start": i, "end": i + chunk_size - 1})
        i += chunk_size
        zone_idx += 1

    widgets_list = []
    x = 0.0
    for z_idx, zone in enumerate(zones):
        for led_i in range(zone['start'], zone['end'] + 1):
            title = zone['name'] if led_i == zone['start'] else ""
            widgets_list.append({
                "title": title,
                "x": x,
                "y": y,
                "width": w,
                "height": h,
                "type": "Single Color View",
                "properties": {
                    "topic": f"{prefix}LED{led_i}",
                    "period": period,
                    "data_type": "string"
                }
            })
            x += w + gap
        x += zone_gap

    # 输出 JSON
    json_str = json.dumps(widgets_list, indent=2)
    with output:
        print(f'生成 {len(widgets_list)} 个 LED 小部件, {len(zones)} 个区域')
        print(f'总宽度: {x - zone_gap + w:.0f}px')
        display(JSON(json_str))

    return widgets_list

# 绑定按钮
ui.children[-2].on_click(generate_layout)

# 同时也实时更新
for w in [w_led_count, w_width, w_height, w_gap, w_zone_gap, w_y_offset, w_period, w_topic_prefix]:
    w.observe(lambda _: generate_layout(), names='value')

# 初始生成一次
generate_layout()

[{'title': 'ZoneA',
  'x': 0.0,
  'y': -65.0,
  'width': 65.0,
  'height': 130.0,
  'type': 'Single Color View',
  'properties': {'topic': '/SmartDashboard/LED/LED0',
   'period': 0.060000000000000005,
   'data_type': 'string'}},
 {'title': '',
  'x': 50.0,
  'y': -65.0,
  'width': 65.0,
  'height': 130.0,
  'type': 'Single Color View',
  'properties': {'topic': '/SmartDashboard/LED/LED1',
   'period': 0.060000000000000005,
   'data_type': 'string'}},
 {'title': '',
  'x': 100.0,
  'y': -65.0,
  'width': 65.0,
  'height': 130.0,
  'type': 'Single Color View',
  'properties': {'topic': '/SmartDashboard/LED/LED2',
   'period': 0.060000000000000005,
   'data_type': 'string'}},
 {'title': '',
  'x': 150.0,
  'y': -65.0,
  'width': 65.0,
  'height': 130.0,
  'type': 'Single Color View',
  'properties': {'topic': '/SmartDashboard/LED/LED3',
   'period': 0.060000000000000005,
   'data_type': 'string'}},
 {'title': '',
  'x': 200.0,
  'y': -65.0,
  'width': 65.0,
  'height': 130.0,
  'type': '

In [ ]:
# ========== 保存到文件 ==========
import os

def save_layout(btn=None):
    widgets_list = generate_layout()
    if not widgets_list:
        return

    # 读取现有布局，合并 LED widgets
    layout_path = 'elastic-layout.json'
    if os.path.exists(layout_path):
        with open(layout_path, 'r') as f:
            layout = json.load(f)
    else:
        layout = {"version": 1.0, "grid_size": 65, "tabs": []}

    # 找到 Teleoperated_DashBoard tab
    tab = None
    for t in layout.get('tabs', []):
        if t.get('name') == 'Teleoperated_DashBoard':
            tab = t
            break

    if tab is None:
        print('未找到 Teleoperated_DashBoard tab')
        return

    # 移除旧的 LED widgets
    containers = tab['grid_layout']['containers']
    containers = [
        c for c in containers
        if not (c.get('type') == 'Single Color View' and
                c.get('properties', {}).get('topic', '').startswith(
                    w_topic_prefix.value + 'LED'))
    ]
    containers.extend(widgets_list)
    tab['grid_layout']['containers'] = containers

    with open(layout_path, 'w') as f:
        json.dump(layout, f, indent=2)

    with output:
        print(f'已保存到 {layout_path}')

save_btn = widgets.Button(
    description='保存到 elastic-layout.json',
    button_style='success',
    layout=widgets.Layout(width='200px')
)
save_btn.on_click(save_layout)
display(save_btn)